# Baseline 1 — TabTransformer on ISCX-URL2016
**Mục tiêu:** 29 tabular features → TabTransformer → binary classification  
**Dataset:** `iscxurl2016` → `ISCXURL2016.csv`  
**Thời gian:** ~15–30 phút trên GPU T4

In [ ]:
import os, sys, json, re, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
)
from tqdm.notebook import tqdm

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})
warnings.filterwarnings('ignore')

SEED = 42; N_FOLDS = 5
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
KAGGLE_INPUT = Path('/kaggle/input')
PROJECT = Path('..')
RAW_DIR = PROJECT / 'data' / 'raw'
OUT_DIR = Path('/kaggle/working') if KAGGLE_INPUT.exists() else PROJECT / 'data'
MODEL_DIR = OUT_DIR / 'models'; FIG_DIR = OUT_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect ISCX URL2016 CSV anywhere under /kaggle/input/
iscx_csv = RAW_DIR / 'ISCXURL2016.csv'
if KAGGLE_INPUT.exists():
    for p in KAGGLE_INPUT.rglob('ISCXURL2016.csv'):
        iscx_csv = p; break
print(f'ISCX: {iscx_csv.exists()} -> {iscx_csv}')

In [ ]:
ISCX_FEATURES_NUM = [
    'urlLen','domainlength','pathLength','subDirLen','fileNameLen',
    'this.fileExtLen','ArgLen','Entropy_URL','Entropy_Domain',
    'Entropy_DirectoryName','Entropy_Filename','Entropy_Afterpath',
    'spcharUrl','URL_DigitCount','host_DigitCount','NumberRate_URL',
    'NumberRate_Domain','NumberRate_DirectoryName','NumberRate_FileName',
    'SymbolCount_URL','SymbolCount_Domain','URL_Letter_Count',
    'host_letter_count','NumberofDotsinURL','LongestPathTokenLength',
    'CharacterContinuityRate','Domain_LongestWordLength',
]
ISCX_FEATURES_CAT = ['URL_sensitiveWord', 'ISIpAddressInDomainName']
TABULAR_DIM = 29

df = pd.read_csv(iscx_csv, encoding='utf-8', low_memory=False)
print(f'Rows: {len(df)}, Cols: {len(df.columns)}')

avail_num = [c for c in ISCX_FEATURES_NUM if c in df.columns]
avail_cat = [c for c in ISCX_FEATURES_CAT if c in df.columns]
print(f'Numerical: {len(avail_num)}/{len(ISCX_FEATURES_NUM)}, Categorical: {len(avail_cat)}/{len(ISCX_FEATURES_CAT)}')

# Numerical — RAW values; StandardScaler is fitted per-fold inside CV (leakage-safe)
X_num = df[avail_num].copy().replace([np.inf, -np.inf], np.nan)
for c in X_num.columns: X_num[c] = pd.to_numeric(X_num[c], errors='coerce')
X_num = X_num.fillna(0).astype(np.float32)

# Categorical — integer labels, NOT standardized
X_cat = df[avail_cat].fillna(0).astype(int).clip(lower=0) if avail_cat else None

y = (df['URL_Type_obf_Type'].astype(str).str.strip().str.lower() == 'phishing').astype(int).values
print(f'Phishing: {y.sum()}, Benign: {(y==0).sum()}')
print(f'Raw num: {X_num.shape}, cat: {None if X_cat is None else X_cat.shape}')

# ── Dataset statistics (for thesis figures) ──
dataset_stats = {
    'dataset': 'ISCX-URL2016',
    'n_samples': int(len(df)),
    'n_phishing': int(y.sum()),
    'n_benign': int((y==0).sum()),
    'phishing_ratio': round(float(y.mean()), 6),
    'n_features': len(avail_num) + (len(avail_cat) if avail_cat else 0),
}
with open(OUT_DIR / 'dataset_stats_iscx.json', 'w') as f:
    json.dump(dataset_stats, f, indent=2)
print(f'Stats saved to {OUT_DIR / "dataset_stats_iscx.json"}')

In [ ]:
class FeatureEmbedding(nn.Module):
    def __init__(self, d=32): super().__init__(); self.e = nn.Linear(1, d)
    def forward(self, x): return self.e(x.unsqueeze(-1))

class TabTransformer(nn.Module):
    def __init__(self, nf=29, ed=32, nh=4, hd=256, od=128, dp=0.1):
        super().__init__()
        self.embs = nn.ModuleList([FeatureEmbedding(ed) for _ in range(nf)])
        self.attn = nn.MultiheadAttention(ed, nh, batch_first=True, dropout=dp)
        self.n1 = nn.LayerNorm(ed); self.n2 = nn.LayerNorm(ed)
        self.ff = nn.Sequential(nn.Linear(ed, hd), nn.GELU(), nn.Dropout(dp), nn.Linear(hd, ed), nn.Dropout(dp))
        self.proj = nn.Linear(ed * nf, od)
        self.cls = nn.Sequential(nn.Linear(od, 64), nn.ReLU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        h = torch.stack([e(x[:, i]) for i, e in enumerate(self.embs)], 1)
        a, _ = self.attn(h, h, h); h = self.n1(h + a)
        f = self.ff(h); h = self.n2(h + f)
        return self.cls(self.proj(h.reshape(h.size(0), -1)))

In [ ]:
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X) if isinstance(X, np.ndarray) else X
        self.y = torch.from_numpy(y.reshape(-1,1).astype(np.float32)) if isinstance(y, np.ndarray) else y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

def compute_metrics(labels, preds):
    pb = (preds >= 0.5).astype(int)
    fpr_val = 0.0
    if len(np.unique(labels)) > 1:
        cm  = confusion_matrix(labels, pb)
        tn, fp = cm[0, 0], cm[0, 1]
        fpr_val = round(fp / max(tn + fp, 1), 4)
    return {'accuracy': accuracy_score(labels, pb),
            'precision': precision_score(labels, pb, zero_division=0),
            'recall': recall_score(labels, pb, zero_division=0),
            'f1': f1_score(labels, pb, zero_division=0),
            'auc': roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.0,
            'fpr': fpr_val}

def train_epoch(model, loader, opt, crit):
    model.train(); total = 0
    for Xb, yb in loader:
        Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad(); loss = crit(model(Xb), yb)
        loss.backward(); opt.step(); total += loss.item() * Xb.size(0)
    return total / len(loader.dataset)

def evaluate(model, loader, crit):
    model.eval(); total = 0; preds, labs = [], []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            logits = model(Xb)
            total += crit(logits, yb).item() * Xb.size(0)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            labs.extend(yb.cpu().numpy())
    preds, labs = np.array(preds), np.array(labs)
    m = compute_metrics(labs, preds); m['loss'] = total / len(loader.dataset)
    return m, preds, labs

In [ ]:
BS, EP, LR = 64, 50, 1e-3
pos_weight = torch.tensor([(len(y) - y.sum()) / y.sum()], device=DEVICE)
crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f'pos_weight={pos_weight.item():.2f} (neg={int(len(y)-y.sum())}, pos={int(y.sum())})')

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
all_metrics, all_preds, all_labels = [], [], []
folds_meta, history = [], []

def build_features(num, cat):
    """Concatenate (per-fold scaled) numeric + raw categorical, pad to 29."""
    feat = np.concatenate([num, cat], axis=1) if cat is not None else num
    if feat.shape[1] < TABULAR_DIM:
        pad = np.zeros((feat.shape[0], TABULAR_DIM - feat.shape[1]), dtype=np.float32)
        feat = np.concatenate([feat, pad], axis=1)
    return feat.astype(np.float32)

def fold_cat(idx):
    return None if X_cat is None else X_cat.values.astype(np.float32)[idx]

X_num_v = X_num.values.astype(np.float32)

for fold, (tr_idx, te_idx) in enumerate(skf.split(np.arange(len(y)), y)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')
    # Per-fold StandardScaler: fit on train fold ONLY, then transform test fold
    scaler = StandardScaler().fit(X_num_v[tr_idx])
    X_tr = build_features(scaler.transform(X_num_v[tr_idx]).astype(np.float32), fold_cat(tr_idx))
    X_te = build_features(scaler.transform(X_num_v[te_idx]).astype(np.float32), fold_cat(te_idx))
    y_tr, y_te = y[tr_idx], y[te_idx]
    tr_ld = DataLoader(SimpleDataset(X_tr, y_tr), batch_size=BS, shuffle=True)
    te_ld = DataLoader(SimpleDataset(X_te, y_te), batch_size=BS)
    model = TabTransformer(nf=X_tr.shape[1]).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EP)
    hist = []
    for ep in range(1, EP + 1):
        tl = train_epoch(model, tr_ld, opt, crit)
        m, _, _ = evaluate(model, te_ld, crit)
        sched.step()
        hist.append({'epoch': ep, 'train_loss': round(float(tl), 5),
                     'val_auc': round(float(m['auc']), 5), 'val_f1': round(float(m['f1']), 5)})
        if ep % 10 == 0:
            print(f'  Epoch {ep:2d}/{EP} | Loss: {tl:.4f} | AUC: {m["auc"]:.4f} | F1: {m["f1"]:.4f}')
    fm, fp, fl = evaluate(model, te_ld, crit)
    fm['fold'] = fold + 1
    all_metrics.append(fm); all_preds.append(fp); all_labels.append(fl)
    history.append({'fold': fold + 1, 'epochs': hist})
    torch.save(model.state_dict(), MODEL_DIR / f'baseline1_fold{fold+1}.pt')
    folds_meta.append({
        'fold': int(fold + 1),
        'test_indices': te_idx.tolist(),
        'scaler_mean': scaler.mean_.tolist(),
        'scaler_scale': scaler.scale_.tolist(),
        'n_features': int(X_tr.shape[1]),
    })
    print(f'  Done: Acc={fm["accuracy"]:.4f}, AUC={fm["auc"]:.4f}, F1={fm["f1"]:.4f}')

# ── Save artifacts for evaluation/figures ──
with open(MODEL_DIR / 'baseline1_folds.json', 'w') as f:
    json.dump({'n_folds': N_FOLDS, 'folds': folds_meta}, f, indent=2)
with open(OUT_DIR / 'training_logs_baseline1.json', 'w') as f:
    json.dump(history, f, indent=2)
fp_obj = np.empty(N_FOLDS, dtype=object); fl_obj = np.empty(N_FOLDS, dtype=object)
for f in range(N_FOLDS):
    fp_obj[f] = all_preds[f]; fl_obj[f] = all_labels[f]
np.savez(OUT_DIR / 'predictions_baseline1.npz',
         preds=np.concatenate(all_preds), labels=np.concatenate(all_labels),
         fold_preds=fp_obj, fold_labels=fl_obj)
print(f'Artifacts saved: baseline1_folds.json, training_logs_baseline1.json, predictions_baseline1.npz')

avg = {k: np.mean([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
std = {k: np.std([m[k] for m in all_metrics]) for k in ['accuracy','precision','recall','f1','auc','fpr']}
print(f'\n>>> {N_FOLDS}-Fold CV: Acc={avg["accuracy"]:.4f}+-{std["accuracy"]:.4f}, AUC={avg["auc"]:.4f}+-{std["auc"]:.4f}, F1={avg["f1"]:.4f}+-{std["f1"]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
a = axes.flatten()

all_p = np.concatenate(all_preds); all_l = np.concatenate(all_labels)
cm = confusion_matrix(all_l, (all_p >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=a[0],
            xticklabels=['Benign','Phishing'], yticklabels=['Benign','Phishing'])
a[0].set_title('Baseline 1 — Confusion Matrix'); a[0].set_ylabel('True'); a[0].set_xlabel('Predicted')

for f in range(N_FOLDS):
    fpr, tpr, _ = roc_curve(all_labels[f], all_preds[f])
    a[1].plot(fpr, tpr, color=colors[f], lw=1.5, alpha=0.7,
              label=f'Fold {f+1} (AUC={roc_auc_score(all_labels[f], all_preds[f]):.4f})')
fpr, tpr, _ = roc_curve(all_l, all_p)
a[1].plot(fpr, tpr, 'k--', lw=2.5, label=f'Pooled (AUC={roc_auc_score(all_l, all_p):.4f})')
a[1].plot([0,1],[0,1], 'gray', lw=1, alpha=0.5)
a[1].set_title('ROC Curves (5-fold CV)'); a[1].set_xlabel('FPR'); a[1].set_ylabel('TPR')
a[1].legend(fontsize=8, loc='lower right')

names = ['accuracy','precision','recall','f1','auc']
x = np.arange(len(names)); means = [avg[m] for m in names]; stdevs = [std[m] for m in names]
a[2].bar(x, means, yerr=stdevs, capsize=5, color='#1f77b4', alpha=0.8)
a[2].set_xticks(x); a[2].set_xticklabels([m.capitalize() for m in names])
a[2].set_ylim(0, 1); a[2].set_title('Metrics (Mean+-Std)')
for i, (m, s) in enumerate(zip(means, stdevs)):
    a[2].text(i, m + s + 0.02, f'{m:.3f}+-{s:.3f}', ha='center', fontsize=8)

# Training curves — mean across folds
av = []
max_ep = max(len(h['epochs']) for h in history)
for ep in range(1, max_ep + 1):
    rows = [h['epochs'][ep-1] for h in history if len(h['epochs']) >= ep]
    av.append({'epoch': ep, **{k: float(np.mean([r[k] for r in rows])) for k in ['train_loss','val_auc','val_f1']}})
a[3].plot([r['epoch'] for r in av], [r['train_loss'] for r in av], 'o-', color='#d62728', lw=1.5, label='Train loss')
a[3].set_xlabel('Epoch'); a[3].set_ylabel('Loss', color='#d62728')
a[3].tick_params(axis='y', labelcolor='#d62728')
a3b = a[3].twinx()
a3b.plot([r['epoch'] for r in av], [r['val_auc'] for r in av], 's-', color='#1f77b4', lw=1.5, label='Val AUC')
a3b.plot([r['epoch'] for r in av], [r['val_f1'] for r in av], 'd-', color='#2ca02c', lw=1.5, label='Val F1')
a3b.set_ylim(0, 1); a3b.set_ylabel('Score')
a3b.legend(fontsize=8, loc='lower left')
a[3].set_title('Training Curves (mean across folds)')

# Data distribution
bars = a[4].bar(['Benign','Phishing'], [dataset_stats['n_benign'], dataset_stats['n_phishing']],
                color=['#2ca02c','#d62728'], alpha=0.85)
a[4].set_ylabel('Samples'); a[4].set_title(f"ISCX-URL2016 Distribution (n={dataset_stats['n_samples']:,})")
for b, v in zip(bars, [dataset_stats['n_benign'], dataset_stats['n_phishing']]):
    a[4].text(b.get_x() + b.get_width()/2, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
a[4].text(0.5, -0.12, f"Ratio {dataset_stats['phishing_ratio']:.2%} phishing",
          transform=a[4].transAxes, ha='center', fontsize=9)

a[5].axis('off')
a[5].text(0.02, 0.95, 'Baseline 1 — TabTransformer (ISCX-URL2016)', fontsize=13, weight='bold', va='top')
a[5].text(0.02, 0.75, f"5-Fold CV:\nAcc={avg['accuracy']:.4f}+-{std['accuracy']:.4f}\n"
                      f"AUC={avg['auc']:.4f}+-{std['auc']:.4f}\n"
                      f"F1={avg['f1']:.4f}+-{std['f1']:.4f}\n"
                      f"Precision={avg['precision']:.4f}+-{std['precision']:.4f}\n"
                      f"Recall={avg['recall']:.4f}+-{std['recall']:.4f}", fontsize=11, va='top')

plt.tight_layout(); plt.savefig(FIG_DIR / 'baseline1_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {FIG_DIR / "baseline1_summary.png"}')

In [ ]:
results = {'model': 'Baseline 1 - TabTransformer (ISCX)', **avg, **{k + '_std': float(std[k]) for k in std}}
with open(MODEL_DIR / 'evaluation_baseline1.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved to {MODEL_DIR / "evaluation_baseline1.json"}')

---
### Download từ Output tab:
- `figures/baseline1_summary.png` (báo cáo)
- `data/models/evaluation_baseline1.json`
- `data/models/baseline1_folds.json`
- `training_logs_baseline1.json`
- `predictions_baseline1.npz`
- `dataset_stats_iscx.json`
- `data/models/baseline1_fold1..5.pt` (optional)

Sau đó chạy **Baseline 2** → `kaggle_baseline2.ipynb`